In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 3

In [7]:
!pip install -q faiss-cpu sentence-transformers transformers scikit-learn pandas numpy

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

kb = []
for _, row in train.iterrows():
    kb.append(str(row[row["answer"]]))

model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = model.encode(kb, show_progress_bar=False)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.array(kb_embeddings, dtype=np.float32))

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]
ans_150 = str(row_150[row_150["answer"]])

result = zs(prompt_150, candidate_labels=labels_150)

score = dict(zip(result["labels"], result["scores"]))[ans_150]
print(round(score, 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.384


In [8]:
query_embedding = model.encode([prompt_150], show_progress_bar=False)

distances, indices = index.search(np.array(query_embedding, dtype=np.float32), 10)

rank = np.where(indices[0] == 150)[0]

if len(rank) > 0:
    print(rank[0] + 1)
else:
    print("Not found in top 10")

10


In [9]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

retrieved_indices = indices[0]

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

sorted_order = np.argsort(ce_scores)[::-1]

reranked_indices = retrieved_indices[sorted_order]

rank = np.where(reranked_indices == 150)[0]

if len(rank) > 0:
    print(rank[0] + 1)
else:
    print("Not found")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


In [10]:
from transformers import AutoTokenizer

row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

query_embedding = model.encode([prompt_42], show_progress_bar=False)

distances, indices = index.search(np.array(query_embedding, dtype=np.float32), 5)

docs_5 = [kb[i] for i in indices[0]]

context = " ".join(docs_5)

text = f"Context: {context} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(text, truncation=False)

print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [11]:
true_document = kb[150]

rag_prompt = f"Context: {true_document} Question: {prompt_150}"

result = zs(rag_prompt, candidate_labels=labels_150)

score = dict(zip(result["labels"], result["scores"]))[ans_150]

print(round(score, 3))

0.989


In [12]:
wrong_document = kb[999]

adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

result = zs(adversarial_prompt, candidate_labels=labels_150)

score = dict(zip(result["labels"], result["scores"]))[ans_150]

print(round(score, 3))

0.529


In [13]:
hits = 0

for i in range(100):
    prompt = str(train.iloc[i]["prompt"])
    correct_doc = kb[i]

    query_embedding = model.encode([prompt], show_progress_bar=False)

    distances, indices = index.search(np.array(query_embedding, dtype=np.float32), 5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    if any(correct_doc in doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate, 1))

73.0


In [14]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

map3_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    correct_letter = row["answer"]

    query_embedding = model.encode([prompt], show_progress_bar=False)

    distances, indices = index.search(np.array(query_embedding, dtype=np.float32), 5)

    retrieved_indices = indices[0]
    docs = [kb[idx] for idx in retrieved_indices]

    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    result = zs(rag_prompt, candidate_labels=labels)

    label_to_letter = {labels[j]: chr(ord("A") + j) for j in range(5)}
    ranked_letters = [label_to_letter[label] for label in result["labels"]]

    top3 = ranked_letters[:3]

    if correct_letter in top3:
        map3 = 1 / (top3.index(correct_letter) + 1)
    else:
        map3 = 0

    map3_scores.append(map3)

print(round(np.mean(map3_scores), 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
